# Systematic Review
**References:**
* ❌ too old - https://github.com/chandraveshchaudhari/systematic-reviewpy
* ❌ agentic AI - https://github.com/PouriaRouzrokh/LatteReview
* ✅ Uses PubMed API - https://github.com/gijswobben/pymed

**TO-DO**
* ✅ Clean this notebook up
* Run the query on other databases as well
* QC the results from each database, starting with pubmed.
* Once QC'd, merge the results across each database result and process.

**REFERENCES THAT CAUGHT MY EYE**
* https://pubmed.ncbi.nlm.nih.gov/33202965/
* https://pubmed.ncbi.nlm.nih.gov/38389433/

In [1]:
# Import all required packages
from datetime import datetime
from dotenv import load_dotenv
from pathlib import Path
from pymed import PubMed
import json
import os
import pandas as pd

# Load environment variables
load_dotenv()

True

# Specify Search Strategy

In [ ]:
# Create a GraphQL query in plain text
# Hack: I used the query builder on https://pubmed.ncbi.nlm.nih.gov/advanced/ to create this

# This is the bottom-up query that guided the keywords I wanted.
#query = '"(geospatial analysis) AND (parkinson* disease) NOT ((motor) OR (global burden))"' # results = 6

# These are big-to-small queries.
#query = "((geospatial analysis) OR (geographic analysis)) AND (parkinson* disease) AND (environmental atmospheric)"
#query = "(parkinson* disease[title]) AND ((geospatial analysis) OR (pollution)) NOT ((motor) OR (genetic) OR (neurologic) OR (nicotine))" # results = 149
#query = "(parkinson* disease[title]) AND ((geospatial analysis) OR (pollution)) NOT ((motor) OR (genetic) OR (neurologic) OR (nicotine) OR (smoking))" # results = 129

# This is the final query.
query = "(parkinson* disease[title]) AND ((geospatial analysis) OR (pollution)) NOT ((motor) OR (genetic) OR (neurologic) OR (nicotine) OR (smoking) OR (halitosis) OR (disease-like) OR (treatment[title]) OR (neuroprotective[title]) OR (maternal) OR (preventative) OR (therapy))" # results = 86

# Query PubMed

In [2]:
# Create a PubMed object that GraphQL can use to query
# Note that the parameters are not required but kindly requested by PubMed Central
# https://www.ncbi.nlm.nih.gov/pmc/tools/developers/
pubmed = PubMed(tool="MyTool", email=os.getenv("PUBMED_EMAIL"))

In [5]:
# Execute the query against the API
# Convert to list so the results can be iterated multiple times (not exhausted)
results = list(pubmed.query(query, max_results=100))

# Export Query Results

In [7]:
# Config
MAX_RESULTS = 100
OUT_DIR = Path("pubmed_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
# Ensure `results` is available and not an exhausted iterator
try:
    has_items = hasattr(results, "__len__") and len(results) > 0
except NameError:
    has_items = False

if not has_items:
    print("`results` is empty or undefined — running the query to fetch items")
    # Re-run the query and store as list so it can be reused
    results = list(pubmed.query(query, max_results=MAX_RESULTS))

In [9]:
# Timestamped filename to avoid accidental overwrites
timestamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
OUT = OUT_DIR / f"results-{timestamp}.ndjson"
QUERY_FILE = OUT_DIR / f"query-{timestamp}.txt"

In [10]:
def article_to_dict(article):
    # Prefer the object's own JSON if available
    try:
        raw = article.toJSON()
        if isinstance(raw, str):
            return json.loads(raw)
        if isinstance(raw, dict):
            return raw
    except Exception:
        pass

    # Fallback: extract common fields safely
    return {
        "pubmed_id": getattr(article, "pubmed_id", None),
        "title": getattr(article, "title", None),
        "keywords": [k for k in (getattr(article, "keywords", []) or []) if k],
        "publication_date": str(getattr(article, "publication_date", "") or ""),
        "abstract": getattr(article, "abstract", None),
    }

count = 0
with OUT.open("w", encoding="utf-8") as fh:
    for a in results:
        try:
            obj = article_to_dict(a)
            fh.write(json.dumps(obj, ensure_ascii=False))
            fh.write("\n")
            count += 1
        except Exception as e:
            # log and continue
            print(f"Failed to write article {getattr(a, 'pubmed_id', '<unknown>')}: {e}")

In [11]:
# Write the query metadata and query string to a timestamped text file
try:
    with QUERY_FILE.open("w", encoding="utf-8") as qf:
        qf.write(f"timestamp: {timestamp}\n")
        qf.write(f"max_results: {MAX_RESULTS}\n")
        qf.write("query:\n")
        qf.write(query)
except Exception as e:
    print(f"Failed to write query file: {e}")

print(f"Wrote {count} records to {OUT.resolve()}")
print(f"Wrote query file to {QUERY_FILE.resolve()}")

Wrote 86 records to /mnt/c/Users/ReginaChua/Desktop/sysrev/pubmed_results/results-20251030T083228Z.ndjson
Wrote query file to /mnt/c/Users/ReginaChua/Desktop/sysrev/pubmed_results/query-20251030T083228Z.txt


# Get a list of articles
Convert the latest NDJSON to a GitHub-friendly CSV (saved in project root)

In [13]:
# Configure paths - NDJSON in pubmed_results/, CSV in project root
OUT_DIR = Path('pubmed_results')
csv_name = Path(f'results-{datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")}.csv')

In [14]:
# Re-use the DataFrame if it exists, otherwise load from NDJSON
if 'ndjson_df' not in globals():
    # Find latest NDJSON
    files = sorted(OUT_DIR.glob('results*.ndjson'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not files:
        raise FileNotFoundError("No NDJSON files found. Run the export cell first.")
    
    latest = files[0]
    print(f"Loading from {latest.name}")
    
    # Load NDJSON
    records = []
    with latest.open('r', encoding='utf-8') as fh:
        for line in fh:
            if line.strip():
                records.append(json.loads(line))
    
    # Create DataFrame
    ndjson_df = pd.json_normalize(records)
    
    # Convert keywords to strings if present
    if 'keywords' in ndjson_df.columns:
        ndjson_df['keywords'] = ndjson_df['keywords'].apply(lambda k: ', '.join(k) if isinstance(k, (list, tuple)) else k)


In [15]:
# Save as CSV with minimal processing for GitHub readability
try:
    # Reorder columns for readability (put common fields first)
    preferred = ['pubmed_id', 'title', 'publication_date', 'keywords', 'abstract']
    cols = [c for c in preferred if c in ndjson_df.columns] + [c for c in ndjson_df.columns if c not in preferred]
    
    # Write CSV (UTF-8 encoding, no index) to project root
    ndjson_df[cols].to_csv(csv_name, index=False, encoding='utf-8')
    print(f"Saved CSV to project root: {csv_name}")
    
except Exception as e:
    print(f"Error saving CSV: {e}")

Saved CSV to project root: results-20251030T083636Z.csv
